<a href="https://colab.research.google.com/github/rimshaejaz123/CV-assignment/blob/main/datasetwork.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt -O KDDTrain+.TXT

--2026-10-02 06:28:12--  https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19109424 (18M) [text/plain]
Saving to: ‘KDDTrain+.TXT’

KDDTrain+.TXT       100%[===================>]  18.22M  --.-KB/s    in 0.1s    

2026-10-02 06:28:13 (153 MB/s) - ‘KDDTrain+.TXT’ saved [19109424/19109424]



In [ ]:
import pandas as pd

columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land',
    'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count',
    'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate',
    'attack_type', 'difficulty_level'
]

df = pd.read_csv('KDDTrain+.TXT', names=columns)

# 3. Verify Lab Requirements
print("Dataset Shape:", df.shape)
print("Lab Requirements Met?:", df.shape[0] >= 100000 and len(df.columns) >= 35)
display(df.head(3))

Dataset Shape: (125973, 43)
Lab Requirements Met?: True


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,attack_type,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0,neptune,19


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, roc_auc_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 1. Preprocess the Data
# Drop 'difficulty_level' as it is just metadata
df = df.drop('difficulty_level', axis=1)

# Convert to Binary Classification: 0 for 'normal', 1 for any attack
df['attack_type'] = df['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)

# Convert text categories (like TCP, UDP) into numbers
categorical_cols = ['protocol_type', 'service', 'flag']
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# 2. Split into Features (X) and Labels (y)
X = df.drop('attack_type', axis=1)
y = df['attack_type']

# Scale large numbers (like bytes) to a uniform range
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 3. Train the Models
print("Training Decision Tree...")
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
dt_probs = dt.predict_proba(X_test)[:, 1]

print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_probs = rf.predict_proba(X_test)[:, 1]

# 4. Generate Required Lab Metrics
def print_metrics(name, y_true, y_pred, y_prob):
    print(f"\n=== {name} Performance ===")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_true, y_prob):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

print_metrics("Decision Tree", y_test, dt_preds, dt_probs)
print_metrics("Random Forest", y_test, rf_preds, rf_probs)

Training Decision Tree...
Training Random Forest...

=== Decision Tree Performance ===
Accuracy:  0.9978
Precision: 0.9981
F1-Score:  0.9977
AUC-ROC:   0.9979
Confusion Matrix:
 [[13400    22]
 [   33 11740]]

=== Random Forest Performance ===
Accuracy:  0.9986
Precision: 0.9994
F1-Score:  0.9985
AUC-ROC:   1.0000
Confusion Matrix:
 [[13415     7]
 [   28 11745]]
